# 🚀 Модуль 2. Урок 7. Prompt Engineering для SQL: генерация сложных запросов и защита от галлюцинаций

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_02_sql/lesson_07_prompt_engineering_for_sql_and_validation.ipynb)

## 📌 Бизнес-контекст и методология
В 2027 году аналитики генерируют SQL с помощью LLM, но проводят обязательную валидацию схемы и плана выполнения.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

db_path = ensure_dataset("ecommerce.db")
conn = sqlite3.connect(db_path)
print(f"🟢 Подключение к базе данных успешно установлено: {db_path}")


🟢 Подключение к базе данных успешно установлено: course\datasets\ecommerce.db


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(ensure_dataset("ecommerce.db"))

def validate_and_explain_sql(query, connection):
    """Проверяет синтаксис запроса через EXPLAIN QUERY PLAN."""
    try:
        explain_query = f"EXPLAIN QUERY PLAN {query}"
        plan = pd.read_sql_query(explain_query, connection)
        print("🟢 План выполнения запроса:")
        print(plan)
        return True
    except Exception as e:
        print(f"🔴 Ошибка в сгенерированном SQL: {e}")
        return False

test_query = """
SELECT 
    customer_id, 
    COUNT(order_id) AS cnt
FROM orders
WHERE status = 'Completed'
GROUP BY customer_id
HAVING cnt > 2;
"""

validate_and_explain_sql(test_query, conn)


🟢 План выполнения запроса:
   id  parent  notused                        detail
0   6       0      216                   SCAN orders
1  10       0        0  USE TEMP B-TREE FOR GROUP BY


True

### 🎯 Практическое задание 1: Обнаружение галлюцинации схемы
Проверьте реальные колонки таблицы `customers` и составьте валидный запрос.

In [3]:
### ВАШ КОД ЗДЕСЬ
pragma_df = pd.read_sql_query("PRAGMA table_info(customers);", conn)
print("Реальные колонки таблицы customers:")
print(pragma_df[['name', 'type']])

corrected_query = """
SELECT 
    customer_id, 
    first_name, 
    city,
    registration_date
FROM customers
LIMIT 5;
"""

assert validate_and_explain_sql(corrected_query, conn), "Запрос должен быть валидным"
print("✅ Тест успешно пройден: галлюцинация схемы устранена!")


Реальные колонки таблицы customers:
                  name     type
0          customer_id  INTEGER
1           first_name     TEXT
2                 city     TEXT
3  acquisition_channel     TEXT
4    registration_date     TEXT
5               device     TEXT
🟢 План выполнения запроса:
   id  parent  notused          detail
0   3       0      216  SCAN customers
✅ Тест успешно пройден: галлюцинация схемы устранена!


### 💡 Шаблон промпта для SQL
Всегда передавайте в промпт DDL схему (`CREATE TABLE` или `PRAGMA table_info`) и требование фильтрации `status = 'Completed'`.